In [0]:
select * from gizmobox.bronze.v_orders;

In [0]:
select 
value:items[0] as item_1,
value:items[1] as item_2
from gizmobox.bronze.v_orders

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
select
--fixing string value field  
regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') AS fixed_value
from gizmobox.bronze.v_orders;

In [0]:
select * from tv_orders_fixed

In [0]:
select schema_of_json(fixed_value)
from tv_orders_fixed
LIMIT 1;


In [0]:
Create table gizmobox.silver.orders_json
AS
select from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed;

In [0]:
select * from gizmobox.silver.orders_json;

In [0]:
Create TEMP VIEW tv_orders_exploded
AS
select
json_value.order_id as order_id,
json_value.order_date as order_date,
json_value.customer_id as customer_id,
json_value.payment_method as payment_method,
json_value.total_amount as total_amount,
json_value.order_status as order_status,
json_value.transaction_timestamp as transaction_timestamp,
explode(array_distinct(json_value.items)) as item
from gizmobox.silver.orders_json;


In [0]:
select * from tv_orders_exploded;

In [0]:
Create or replace table gizmobox.silver.orders
AS
select 
order_id,
order_date,
customer_id,
payment_method,
total_amount,
order_status,
transaction_timestamp,
item.category as item_category,
item.item_id as item_id,
item.name as item_name,
item.price as item_price,
item.quantity as item_quantity,
item.details.brand as item_brand,
item.details.color as item_color
from tv_orders_exploded; 

In [0]:
select * from gizmobox.silver.orders;